# Stage B — Criterion Annotation with DeepSeek-R1-Distill-Qwen-32B

Generates `(patient, criterion) → label + reasoning` training annotations for the
TREC21+KZ training split using DeepSeek-R1-Distill-Qwen-32B (float16, on-device).
Output feeds Stage C fine-tuning of an open criterion cross-encoder.

**Run order matters — do not skip or reorder cells.**
The symlink fixing vLLM's CUDA mismatch must be created before `import vllm`.

## Dependencies

In [ ]:
import subprocess, sys
# vLLM is incompatible with this Colab environment (CUDA 12 / Pillow conflicts).
# Using transformers generate() instead — 7B fits in 80GB with large batches.
print('Using transformers backend (vLLM skipped)')

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q sentence-transformers datasets scikit-learn transformers accelerate tqdm
!pip install -q sympy==1.13.1

In [ ]:
import torch
print(f'PyTorch CUDA : {torch.version.cuda}')
print(f'GPU          : {torch.cuda.get_device_name(0)}')

In [ ]:
import os
os.environ['HF_TOKEN'] = ''  # HuggingFace READ token

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_ROOT      = '/content/drive/MyDrive/ct_data23'
QRELS_PATH     = f'{DATA_ROOT}/unified_qrels.jsonl'
CRITERIA_PATH  = f'{DATA_ROOT}/criteria_data.jsonl'
OUTPUT_PATH    = f'{DATA_ROOT}/criteria_r1_labels.jsonl'

CLF_CHECKPOINT = 'semaj83/ctmatch-clf-v4'
MODEL_ID       = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B'

TOP_K_ANNOTATE = 10
MAX_NEW_TOKENS = 800
GEN_BATCH_SIZE = 128  # 7B=14GB + KV cache for 128×1568 tokens ≈ 31GB; well within A100 80GB

TRAIN_SOURCES  = {'trec21', 'kz'}

In [ ]:
import json

topic2text   = {}
topic2source = {}
topic2rel    = {}

with open(QRELS_PATH) as f:
    for line in f:
        rec = json.loads(line)
        if rec['source'] not in TRAIN_SOURCES:
            continue
        tid = rec['topic_id']
        topic2text[tid]   = rec['topic_text']
        topic2source[tid] = rec['source']
        if tid not in topic2rel:
            topic2rel[tid] = {}
        topic2rel[tid][rec['doc_id']] = rec['label']

src_counts = {}
for s in topic2source.values():
    src_counts[s] = src_counts.get(s, 0) + 1
print(f'Training topics: {len(topic2text)}  {src_counts}')

In [ ]:
from datasets import load_dataset

index2docid_ds = load_dataset('semaj83/ctmatch_ir', data_files='index2docid.txt', split='train')
doc_texts_ds   = load_dataset('semaj83/ctmatch_ir', data_files='doc_texts.txt',   split='train')

index2docid = [row['text'].strip() for row in index2docid_ds]
docid2text  = {nct_id: doc_texts_ds[idx]['text'] for idx, nct_id in enumerate(index2docid)}
print(f'Corpus: {len(index2docid):,}')

In [ ]:
nctid2crit = {}
with open(CRITERIA_PATH) as f:
    for line in f:
        rec = json.loads(line)
        nctid2crit[rec['nct_id']] = {
            'include_criteria': rec['include_criteria'],
            'exclude_criteria': rec['exclude_criteria'],
        }
print(f'Criteria loaded for {len(nctid2crit):,} trials')

## Stage 1 — clf-v4 top-K selection
Score judged docs per topic, then free clf-v4 before loading R1.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

clf_tokenizer = AutoTokenizer.from_pretrained(CLF_CHECKPOINT)
clf_model     = AutoModelForSequenceClassification.from_pretrained(CLF_CHECKPOINT)
clf_model.eval().cuda()

id2label     = clf_model.config.id2label
relevant_col = [int(k) for k, v in id2label.items() if v == 'relevant'][0]
print(f'clf-v4 loaded  |  relevant_col={relevant_col}')

In [ ]:
from tqdm.auto import tqdm

def clf_batch_score(topic_text, doc_texts, batch_size=64):
    scores = []
    for i in range(0, len(doc_texts), batch_size):
        pairs = [(topic_text, dt) for dt in doc_texts[i:i+batch_size]]
        enc   = clf_tokenizer(pairs, padding=True, truncation=True,
                              max_length=512, return_tensors='pt').to(clf_model.device)
        with torch.no_grad():
            logits = clf_model(**enc).logits
        scores.extend(F.softmax(logits, dim=1)[:, relevant_col].cpu().tolist())
    return scores

clf_topic2ranked = {}
for tid in tqdm(topic2text, desc='clf-v4 scoring'):
    nct_ids = [nid for nid in topic2rel[tid] if nid in docid2text]
    texts   = [docid2text[nid] for nid in nct_ids]
    scores  = clf_batch_score(topic2text[tid], texts)
    clf_topic2ranked[tid] = sorted(zip(nct_ids, scores), key=lambda x: x[1], reverse=True)

n_criteria = sum(
    len(nctid2crit.get(nid, {}).get('include_criteria', [])) +
    len(nctid2crit.get(nid, {}).get('exclude_criteria', []))
    for tid in topic2text
    for nid, _ in clf_topic2ranked[tid][:TOP_K_ANNOTATE]
)
print(f'Total criterion assessments: {n_criteria:,}')

In [ ]:
import gc
del clf_model, clf_tokenizer, docid2text
gc.collect()
torch.cuda.empty_cache()
print('clf-v4 freed')

## Stage 2 — DeepSeek-R1-Distill-Qwen-7B (transformers)\n",
    "\n",
    "7B × 2 bytes = 14GB float16, leaving 66GB free for large batches.\n",
    "At batch_size=32 on A100 80GB: ~45 min for the full training split."

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

r1_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side='left')
r1_tokenizer.pad_token = r1_tokenizer.eos_token

r1_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
r1_model.eval()
print(f'R1-Distill-7B loaded (float16)  |  max_new_tokens={MAX_NEW_TOKENS}')

In [ ]:
import re

VALID_LABELS = {'included', 'not_included', 'excluded', 'not_excluded', 'not_enough_information'}

def make_prompt(patient_text, criterion, crit_type):
    user_msg = (
        f'Patient: {patient_text}\n\n'
        f'{crit_type.capitalize()} criterion: {criterion}\n\n'
        'Assess whether this criterion applies to the patient.\n'
        'End your response with one label on its own line:\n'
        'included / not_included / excluded / not_excluded / not_enough_information'
    )
    return r1_tokenizer.apply_chat_template(
        [{'role': 'user', 'content': user_msg}],
        tokenize=False, add_generation_prompt=True,
    )

def generate_batch(prompts, batch_size=GEN_BATCH_SIZE):
    all_texts = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        enc = r1_tokenizer(batch, return_tensors='pt', padding=True,
                           truncation=True, max_length=768).to(r1_model.device)
        with torch.no_grad():
            out = r1_model.generate(
                **enc,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=r1_tokenizer.eos_token_id,
            )
        input_len = enc['input_ids'].shape[1]
        for seq in out:
            all_texts.append(r1_tokenizer.decode(seq[input_len:], skip_special_tokens=False))
    return all_texts

def parse_label(text):
    cleaned = text.strip().lower().replace(' ', '_').strip('.')
    if cleaned in VALID_LABELS:
        return cleaned
    for label in sorted(VALID_LABELS, key=len, reverse=True):
        if label in cleaned:
            return label
    return 'not_enough_information'

def parse_r1_output(text):
    if '</think>' in text:
        # R1's chat template appends <think> to the prompt, so decoded output
        # starts mid-think-block with no opening tag. Everything before </think>
        # is the reasoning; everything after is the label.
        idx = text.index('</think>')
        reasoning = text[:idx].strip()
        answer    = text[idx + len('</think>'):].strip()
    else:
        # No </think> means the generation was truncated mid-reasoning
        reasoning = text.strip()
        answer    = ''
    for line in reversed(answer.split('\n')):
        line = line.strip()
        if line:
            return parse_label(line), reasoning
    return 'not_enough_information', reasoning

In [ ]:
import textwrap

_topics = list(topic2text.items())
P0, P1, P2 = _topics[0][1], _topics[1][1], _topics[2][1]

SANITY_CASES = [
    ('Karnofsky ≥ 70',  P0, 'inclusion', 'Patients must have a Karnofsky performance status great or equal to 70%'),
    ('Ambulatory',       P1, 'inclusion', 'Patient must be ambulatory and relatively good health. Even if unable to work at least able to partially care for self and not demented'),
    ('Psychiatric excl', P0, 'exclusion', 'Major depression or another major psychiatric disorder as described in DSM IV within the past 2 years'),
    ('Calcium intake',   P2, 'inclusion', 'Calcium intake below a threshold level'),
    ('Corticosteroids',  P1, 'exclusion', 'Have used corticosteroids for more than 30 days within the past 90 days.'),
]

sanity_prompts = [make_prompt(p, c, ct) for _, p, ct, c in SANITY_CASES]
sanity_texts   = generate_batch(sanity_prompts)

print('=== R1-Distill-7B sanity check ===')
for (name, _, crit_type, criterion), raw in zip(SANITY_CASES, sanity_texts):
    label, reasoning = parse_r1_output(raw)
    print(f'\n--- {name} ({crit_type}) ---')
    print(f'Label     : {label}')
    print(f'Reasoning : {textwrap.shorten(reasoning, 200) if reasoning else "(empty)"}')
    print(f'Raw (first 300 chars): {repr(raw[:300])}')

n_trunc = sum(1 for t in sanity_texts if '</think>' not in t)
print(f'\nTruncated: {n_trunc}/{len(SANITY_CASES)}')
nei_count = sum(1 for t in sanity_texts if parse_r1_output(t)[0] == 'not_enough_information')
print(f'NEI: {nei_count}/{len(SANITY_CASES)}')

In [ ]:
# ── Optional repair: strip buggy-parse records from the output file ──────────
#
# Pre-flight in train_criterion_clf.ipynb found 770 records across 8 trec21 topics
# with empty reasoning. These were annotated before the parse fix; the checkpoint
# skipped them on re-run. Their labels may also be wrong (parse_label ran on the
# full raw output including reasoning text).
#
# Running this cell rewrites OUTPUT_PATH without those topics so the annotation
# loop re-annotates them with the correct parse. Safe to skip if already done.

import shutil

def strip_topics(path, bad_topic_ids):
    bad = set(bad_topic_ids)
    tmp = path + '.repair_tmp'
    kept, dropped = 0, 0
    with open(path) as fin, open(tmp, 'w') as fout:
        for line in fin:
            rec = json.loads(line)
            if rec['topic_id'] in bad:
                dropped += 1
            else:
                fout.write(line)
                kept += 1
    shutil.move(tmp, path)
    print(f'Stripped {dropped} records from {len(bad)} topics  |  {kept} records remain')

if os.path.exists(OUTPUT_PATH):
    bad_topics = set()
    with open(OUTPUT_PATH) as f:
        for line in f:
            rec = json.loads(line)
            if not rec.get('reasoning'):
                bad_topics.add(rec['topic_id'])
    if bad_topics:
        print(f'Found {len(bad_topics)} topics with empty reasoning: {sorted(bad_topics)}')
        strip_topics(OUTPUT_PATH, bad_topics)
    else:
        print('No empty-reasoning records found — nothing to repair')

In [ ]:
done_topics = set()
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH) as f:
        for line in f:
            done_topics.add(json.loads(line)['topic_id'])

remaining = [tid for tid in topic2text if tid not in done_topics]
print(f'Already annotated : {len(done_topics)} topics')
print(f'Remaining         : {len(remaining)} topics')

In [ ]:
from collections import Counter

# Collect all prompts upfront across all topics, then generate in one continuous pass.
# This keeps the GPU saturated and avoids per-topic Python/CUDA launch overhead.

all_prompts, all_meta = [], []
for tid in tqdm(remaining, desc='Building prompts'):
    top_k = [nid for nid, _ in clf_topic2ranked[tid][:TOP_K_ANNOTATE]]
    for nid in top_k:
        crit = nctid2crit.get(nid, {'include_criteria': [], 'exclude_criteria': []})
        for c in crit['include_criteria']:
            all_prompts.append(make_prompt(topic2text[tid], c, 'inclusion'))
            all_meta.append((tid, nid, 'inclusion', c))
        for c in crit['exclude_criteria']:
            all_prompts.append(make_prompt(topic2text[tid], c, 'exclusion'))
            all_meta.append((tid, nid, 'exclusion', c))

print(f'Total prompts to generate: {len(all_prompts):,}')

all_texts = generate_batch(all_prompts)

label_counts = Counter()
n_truncated  = 0

with open(OUTPUT_PATH, 'a') as out_f:
    for (tid, nid, crit_type, criterion), raw in tqdm(
            zip(all_meta, all_texts), total=len(all_meta), desc='Writing'):
        label, reasoning = parse_r1_output(raw)
        trunc = '</think>' not in raw
        if trunc:
            n_truncated += 1

        out_f.write(json.dumps({
            'topic_id':  tid,
            'source':    topic2source[tid],
            'nct_id':    nid,
            'crit_type': crit_type,
            'criterion': criterion,
            'label':     label,
            'reasoning': reasoning,
            'truncated': trunc,
        }) + '\n')
        label_counts[label] += 1

total = sum(label_counts.values())
print(f'\nAnnotated {total:,} criteria  |  truncated: {n_truncated} ({100*n_truncated/max(total,1):.1f}%)')
print('\nLabel distribution:')
for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f'  {label:30s} {count:6,}  ({100*count/total:.1f}%)')

In [ ]:
all_recs = []
with open(OUTPUT_PATH) as f:
    for line in f:
        all_recs.append(json.loads(line))

label_dist = Counter(r['label'] for r in all_recs)
total      = len(all_recs)

print(f'Total annotations : {total:,}')
print(f'Unique topics     : {len({r["topic_id"] for r in all_recs})}')
print(f'Unique trials     : {len({r["nct_id"] for r in all_recs})}')
print(f'Source split      : {dict(Counter(r["source"] for r in all_recs))}')
print(f'Truncated         : {sum(1 for r in all_recs if r.get("truncated"))} ({100*sum(1 for r in all_recs if r.get("truncated"))/total:.1f}%)')
print()
for label, count in sorted(label_dist.items(), key=lambda x: -x[1]):
    print(f'  {label:30s} {count:6,}  ({100*count/total:.1f}%)')
nei_pct = 100 * label_dist.get('not_enough_information', 0) / total
print(f'\nNEI rate: {nei_pct:.1f}%  (Claude simple-label baseline: 41.8%)')
print(f'Saved → {OUTPUT_PATH}')